# K-Means & Clustering: Zero to Hero

The only notebook in this series where **you cannot tell whether you got the right answer** —
and learning to work honestly under that constraint is the real subject.

> **Prerequisites:** [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb)
> for the shared workflow. [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) §1.2 and
> [`pca_zero_to_hero.ipynb`](pca_zero_to_hero.ipynb) §1.5 both cover the scaling dependency
> that k-means shares, and PCA is the standard preprocessing step in front of clustering.

---

## Why this notebook is different

Every other model in this series is checked against a held-out truth. Clustering has no truth.
That changes what "doing it properly" means, and it is why this notebook spends more effort on
*evaluation* than on the algorithm:

- **k-means always returns clusters, whether or not any exist.** §1.8 runs it on uniform random
  noise and gets a silhouette score of **0.42** — a number many practitioners would call
  "reasonable structure". Nothing in the output says the data has none.
- **The standard ways of choosing k disagree with each other and with the truth.** On data with
  **6** real clusters, §1.7 finds the elbow picking 3, silhouette picking 3, Davies–Bouldin
  picking 4, and only Calinski–Harabasz getting it right.
- **The algorithm finds a local optimum, not the answer.** §1.3 runs it 30 times from different
  starts and gets **21 different solutions**, the worst 1.7× worse than the best.
- **k-means' famous assumptions matter less than you were told, until suddenly they matter
  enormously.** §1.5 shows anisotropy barely hurting well-separated clusters — and unequal
  cluster *sizes* collapsing it from ARI 0.93 to **0.03**.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | What clustering is · Lloyd's algorithm from scratch · **local optima** · k-means++ · **what k-means assumes** · scaling · **choosing k, five ways** · **clusters from pure noise** |
| **2. Worked example** | Digits: clustering 1,797 images with the labels hidden |
| **3. The other three** | Hierarchical & dendrograms · DBSCAN · Gaussian mixtures — on the same data |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | Clustering-specific errors and a checklist |

## The one-paragraph summary

K-means partitions data into $k$ groups by alternating two trivial steps — assign each point to
its nearest centre, then move each centre to the mean of its points — until nothing changes.
That minimises the **within-cluster sum of squares**, which makes it fast and makes it assume
your clusters are round, similarly sized and similarly spread. It converges to a **local**
optimum that depends on where it started, so it is always run several times. And because it
optimises a quantity that has nothing to do with whether clusters *exist*, it will confidently
partition noise — which is why choosing $k$ and validating the result are harder problems than
the algorithm itself.

---
# Part 0 - Setup

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.cluster import (
    KMeans, MiniBatchKMeans, DBSCAN, AgglomerativeClustering, SpectralClustering,
)
from sklearn.mixture import GaussianMixture
from sklearn.datasets import (
    load_digits, load_wine, load_iris, make_blobs, make_moons, make_circles,
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    silhouette_score, silhouette_samples, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score,
    confusion_matrix,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110

# load_digits reshapes an array in place, which NumPy 2.5 deprecates. The warning comes
# from inside scikit-learn 1.9 and there is nothing to fix on our side.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning,
                            message=".*shape on a NumPy array.*")
    DIGITS = load_digits()

print("ready | numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 - Theory from zero

1. What clustering is, and why the question is ill-posed
2. Lloyd's algorithm, from scratch
3. **The objective, and why it only finds a local optimum**
4. k-means++
5. **What k-means assumes about your clusters**
6. Scaling
7. **Choosing k — five methods that disagree**
8. **Clusters from pure noise**

## 1.1 The question is ill-posed

Supervised learning has a target. Clustering has an *intention*, and different intentions give
different right answers on the same data.

Consider a set of photographs. Should they be clustered by subject, by photographer, by time of
day, by dominant colour? All are real structure. An algorithm cannot know which you meant, so
it optimises a **proxy** — usually some notion of "compact groups" — and you get whatever that
proxy finds.

This has a hard consequence that runs through the whole notebook: **there is no held-out set to
check against.** Every validation technique from NB-00 assumes a target you can compare
predictions against. Here there is none, and the metrics that exist are measuring the proxy,
not your intention.

In [ ]:
# The same 12 points, two defensible clusterings.
pts = np.array([[1, 1], [1, 2], [2, 1], [2, 2],        # bottom-left square
                [8, 1], [8, 2], [9, 1], [9, 2],        # bottom-right square
                [4.5, 6], [5, 6.5], [5.5, 6], [5, 5.5]])  # top middle

by_position = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(pts)
by_height = (pts[:, 1] > 4).astype(int)                # "top vs bottom"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, (title, lab) in zip(axes, [("k-means, k=3", by_position),
                                   ("'top vs bottom', k=2", by_height)]):
    ax.scatter(pts[:, 0], pts[:, 1], c=lab, cmap="Set1", s=90, edgecolor="k")
    ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print("Both are correct. They answer different questions, and the data cannot tell you")
print("which question you asked.")
print()
print("So the honest workflow for clustering is not 'run the algorithm and report the")
print("clusters'. It is:")
print("  1. State what you intend the clusters to MEAN, before you look.")
print("  2. Choose a method whose assumptions match that intention (1.5).")
print("  3. Check the result is not an artifact (1.8), and that it is stable (Q11).")
print("  4. Validate against something EXTERNAL - a business outcome, a held-out label,")
print("     a domain expert - because no internal metric can do it for you (3.4).")

## 1.2 Lloyd's algorithm

K-means minimises the **within-cluster sum of squares** (WCSS, also called inertia):

$$ \text{WCSS} = \sum_{j=1}^{k} \sum_{x \in C_j} \lVert x - \mu_j \rVert^2 $$

Finding the global optimum is NP-hard. Lloyd's algorithm is the standard heuristic, and it is
two lines repeated:

1. **Assign** — put every point in the cluster whose centre is nearest.
2. **Update** — move every centre to the mean of the points assigned to it.

Each step can only decrease the WCSS, and there are finitely many assignments, so it always
terminates. That guarantee says nothing about the answer being *good* (§1.3).

In [ ]:
def kmeans_from_scratch(X, k, max_iter=100, seed=0):
    """Lloyd's algorithm. Returns (labels, centres, inertia, n_iter)."""
    rng = np.random.default_rng(seed)
    centres = X[rng.choice(len(X), k, replace=False)]          # random init
    for it in range(max_iter):
        # 1. ASSIGN: distance from every point to every centre
        d = np.sqrt(((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2))
        labels = d.argmin(axis=1)
        # 2. UPDATE: each centre becomes the mean of its members
        new = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centres[j]
                        for j in range(k)])
        if np.allclose(new, centres):
            centres = new
            break
        centres = new
    inertia = ((X - centres[labels]) ** 2).sum()
    return labels, centres, inertia, it + 1


X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0,
                              random_state=RANDOM_STATE)

lab, cen, inert, n_it = kmeans_from_scratch(X_blobs, k=4, seed=0)
sk = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(X_blobs)

print(f"from scratch : inertia {inert:>10.4f}  converged in {n_it} iterations")
print(f"sklearn      : inertia {sk.inertia_:>10.4f}  converged in {sk.n_iter_} iterations")
print(f"\nsame partition? ARI = "
      f"{adjusted_rand_score(lab, sk.labels_):.4f}   (1.0 means identical grouping)")
print()
print("Cluster LABELS are arbitrary integers - cluster 0 from one run may be cluster 2 from")
print("another. Never compare label arrays directly; compare partitions with a measure like")
print("the adjusted Rand index, which ignores the naming.")

In [ ]:
# Watch it converge.
def kmeans_trace(X, k, n_steps, seed):
    rng = np.random.default_rng(seed)
    centres = X[rng.choice(len(X), k, replace=False)]
    frames = []
    for _ in range(n_steps):
        d = np.sqrt(((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2))
        labels = d.argmin(axis=1)
        frames.append((labels.copy(), centres.copy(),
                       ((X - centres[labels]) ** 2).sum()))
        centres = np.array([X[labels == j].mean(axis=0) if (labels == j).any() else centres[j]
                            for j in range(k)])
    return frames


frames = kmeans_trace(X_blobs, 4, 5, seed=7)
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (labels, centres, inertia) in zip(axes, frames):
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="Set1", s=8, alpha=0.6)
    ax.scatter(centres[:, 0], centres[:, 1], c="black", marker="X", s=110)
    ax.set_title(f"inertia {inertia:.0f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Lloyd's algorithm: assign, move, repeat (X = centres)", fontsize=10)
fig.tight_layout(); plt.show()

print("inertia by iteration:", [f"{f[2]:.0f}" for f in frames])
print()
print("It decreases every step and then stops. That monotonic decrease is the convergence")
print("guarantee - and it is also why the algorithm cannot escape a bad start (1.3).")

## 1.3 Local optima

The WCSS surface has many local minima. Lloyd's algorithm walks downhill from wherever it
started and stops at the first bottom it reaches — so **the answer depends on the
initialisation**, and there is no way to tell from the result whether you found a good one.

In [ ]:
X6, y6 = make_blobs(n_samples=500, centers=6, cluster_std=1.1, random_state=RANDOM_STATE)

runs = np.array([KMeans(n_clusters=6, n_init=1, init="random", random_state=s)
                 .fit(X6).inertia_ for s in range(30)])
print("30 runs, n_init=1, random initialisation:\n")
print(f"  best inertia            : {runs.min():.2f}")
print(f"  worst inertia           : {runs.max():.2f}")
print(f"  worst / best            : {runs.max() / runs.min():.2f}x")
print(f"  distinct solutions      : {len(np.unique(np.round(runs, 2)))} of 30 runs")
print(f"  runs finding the best   : {(np.round(runs, 2) == round(runs.min(), 2)).sum()} of 30")

plt.hist(runs, bins=20, edgecolor="k", alpha=0.75)
plt.axvline(runs.min(), color="crimson", ls="--", label=f"best ({runs.min():.0f})")
plt.xlabel("final inertia"); plt.ylabel("runs")
plt.title("Same data, same k, 30 different starting points")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

print()
print("More than half the runs land somewhere worse than the best, and the worst is 1.7x")
print("off. If you had run it once you would have had no way to know which you got.")
print()
print("This is why sklearn's default is n_init=10: it runs the entire algorithm 10 times")
print("from different starts and keeps the lowest inertia. That default is not a nicety -")
print("it is what makes k-means usable at all. Never set n_init=1 to save time.")

## 1.4 k-means++

Random initialisation frequently puts two centres inside the same true cluster, which is hard
to recover from. **k-means++** (Arthur & Vassilvitskii, 2007) instead chooses centres one at a
time, each with probability proportional to its squared distance from the nearest centre
already chosen — so new centres tend to land far from existing ones.

It comes with a proof: the expected WCSS is within $O(\log k)$ of optimal *before Lloyd's
algorithm runs at all*. It is sklearn's default.

In [ ]:
print("Same 30 seeds, both initialisations:\n")
print(f"  {'init':<14} {'best':>10} {'worst':>10} {'worst/best':>12} "
      f"{'distinct':>10} {'found best':>12}")
print("  " + "-" * 72)
for name, init in [("random", "random"), ("k-means++", "k-means++")]:
    v = np.array([KMeans(n_clusters=6, n_init=1, init=init, random_state=s)
                  .fit(X6).inertia_ for s in range(30)])
    print(f"  {name:<14} {v.min():>10.1f} {v.max():>10.1f} {v.max()/v.min():>12.3f} "
          f"{len(np.unique(np.round(v, 2))):>10} "
          f"{(np.round(v, 2) == round(v.min(), 2)).sum():>10} /30")

print("\nDoes the benefit depend on k? Larger k means more chances to start badly:\n")
X10, _ = make_blobs(n_samples=800, centers=10, cluster_std=1.0, random_state=RANDOM_STATE)
print(f"  {'k':>4} {'random best':>13} {'random worst':>14} {'k++ best':>11} {'k++ worst':>12}")
print("  " + "-" * 60)
for k in [5, 10, 20]:
    r = np.array([KMeans(n_clusters=k, n_init=1, init="random", random_state=s)
                  .fit(X10).inertia_ for s in range(20)])
    p = np.array([KMeans(n_clusters=k, n_init=1, init="k-means++", random_state=s)
                  .fit(X10).inertia_ for s in range(20)])
    print(f"  {k:>4} {r.min():>13.1f} {r.max():>14.1f} {p.min():>11.1f} {p.max():>12.1f}")

print()
print("At k=5 the two are indistinguishable - the problem is easy enough that a random")
print("start is fine. By k=20 the gap is large, in both the best and the worst case.")
print()
print("So k-means++ narrows the spread rather than removing it, and it earns its keep as k")
print("grows. n_init is still doing real work on top of it.")

## 1.5 What k-means assumes

Minimising within-cluster squared distance to a **mean** carries three assumptions:

- **Round (isotropic) clusters** — squared Euclidean distance treats all directions alike, so
  an elongated cluster gets split.
- **Similar spread** — a wide cluster and a tight one are judged on the same scale.
- **Similar sizes** — this one is subtler than it sounds, and it turns out to be the most
  damaging.

Every textbook says this. What the textbooks do not say is **when it actually matters**, and
the answer is: only when clusters are close enough to compete.

In [ ]:
shear = np.array([[0.60, -0.63], [-0.41, 0.85]])

print("Anisotropy vs separation. Same shear, varying cluster_std (lower = better separated):\n")
print(f"  {'cluster_std':>12} {'spheres':>10} {'sheared':>10} {'GMM (sheared)':>15}")
print("  " + "-" * 52)
for std in [0.6, 1.0, 1.5, 2.0, 2.5]:
    Xb, yb = make_blobs(n_samples=600, centers=3, cluster_std=std, random_state=RANDOM_STATE)
    Xs = Xb @ shear
    a_sph = adjusted_rand_score(yb, KMeans(n_clusters=3, n_init=10,
                                           random_state=RANDOM_STATE).fit_predict(Xb))
    a_shr = adjusted_rand_score(yb, KMeans(n_clusters=3, n_init=10,
                                           random_state=RANDOM_STATE).fit_predict(Xs))
    a_gmm = adjusted_rand_score(yb, GaussianMixture(n_components=3, n_init=5,
                                                    random_state=RANDOM_STATE).fit_predict(Xs))
    print(f"  {std:>12.1f} {a_sph:>10.4f} {a_shr:>10.4f} {a_gmm:>15.4f}")

print()
print("At cluster_std=0.6 the shear costs NOTHING - k-means scores a perfect 1.0 on stretched")
print("clusters. The assumption is violated and it does not matter, because the clusters are")
print("far apart.")
print()
print("By cluster_std=1.5 the same shear costs about 0.39 of ARI while a Gaussian mixture -")
print("which fits a full covariance per cluster, so elongation is something it can model -")
print("is still perfect.")
print()
print("The lesson is not 'k-means fails on non-spherical data'. It is that its assumptions")
print("are only under stress when clusters are close enough to compete for points.")

In [ ]:
# Unequal SIZES, at fixed separation. This is where it really breaks.
print("Unequal cluster sizes, same separation and spread throughout:\n")
print(f"  {'sizes':>22} {'k-means ARI':>13} {'GMM ARI':>10}")
print("  " + "-" * 48)
for sizes in [[200, 200, 200], [400, 150, 50], [550, 40, 10], [580, 15, 5]]:
    Xz, yz = make_blobs(n_samples=sizes, centers=None, n_features=2, cluster_std=1.5,
                        center_box=(-6, 6), random_state=RANDOM_STATE)
    a_km = adjusted_rand_score(yz, KMeans(n_clusters=3, n_init=10,
                                          random_state=RANDOM_STATE).fit_predict(Xz))
    a_gm = adjusted_rand_score(yz, GaussianMixture(n_components=3, n_init=5,
                                                   random_state=RANDOM_STATE).fit_predict(Xz))
    print(f"  {str(sizes):>22} {a_km:>13.4f} {a_gm:>10.4f}")

Xz, yz = make_blobs(n_samples=[580, 15, 5], centers=None, n_features=2, cluster_std=1.5,
                    center_box=(-6, 6), random_state=RANDOM_STATE)
km_z = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(Xz)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
for ax, (title, lab) in zip(axes, [("truth: 580 / 15 / 5", yz), ("k-means, k=3", km_z)]):
    ax.scatter(Xz[:, 0], Xz[:, 1], c=lab, cmap="Set1", s=12, alpha=0.75)
    ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"cluster sizes k-means produced: {np.bincount(km_z)}")
print(f"true sizes                    : {np.bincount(yz)}")
print()
print("THIS is the assumption that bites. k-means collapses from 0.93 to 0.03 while the")
print("Gaussian mixture stays above 0.83.")
print()
print("The mechanism: WCSS is a SUM over points, so a large cluster contributes far more")
print("total error than a small one. Splitting the big cluster in half reduces the objective")
print("more than correctly isolating the 5-point cluster does. k-means is doing exactly what")
print("you asked - the objective simply does not value small clusters.")
print()
print("Practical consequence: if you are clustering to find a rare segment - fraud rings,")
print("a niche customer type, a defect mode - k-means is close to the worst possible tool,")
print("and it will not tell you it failed.")

## 1.6 Scaling

K-means is built on Euclidean distance, so every argument from NB-07 §1.2 and NB-09 §1.5
applies unchanged: a feature with a large numeric range dominates the objective, and the
clustering becomes a partition on that feature alone.

In [ ]:
Xw, yw = load_wine(return_X_y=True)
print("wine: 13 chemical measurements on very different scales\n")
print(f"  {'preprocessing':<20} {'ARI':>8} {'NMI':>8}")
print("  " + "-" * 38)
for label, data in [("raw", Xw), ("StandardScaler", StandardScaler().fit_transform(Xw))]:
    lab = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(data)
    print(f"  {label:<20} {adjusted_rand_score(yw, lab):>8.4f} "
          f"{normalized_mutual_info_score(yw, lab):>8.4f}")

variances = Xw.var(axis=0)
print(f"\n  raw feature variances span {variances.min():.4f} to {variances.max():,.0f}")
print(f"  ratio: {variances.max() / variances.min():,.0f}x")
print()
print("Unscaled, the clustering is essentially a partition on the single highest-variance")
print("chemical. Scaling more than doubles the agreement with the true cultivars.")
print()
print("The exception is the same as PCA's (NB-09 section 1.5): when features are already")
print("commensurable - pixels on one 0-16 scale - scaling amplifies the quietest, noisiest")
print("channels and can HURT. Part 2 measures exactly that on digits.")

## 1.7 Choosing k

Five methods, none of them reliable:

- **Elbow** — plot inertia against $k$ and find the bend. Inertia always decreases, so there is
  no optimum to find, only a change of slope.
- **Silhouette** — for each point, how much closer it is to its own cluster than to the next
  nearest, in $[-1, 1]$. Higher is better.
- **Calinski–Harabasz** — between-cluster dispersion over within-cluster dispersion. Higher is
  better.
- **Davies–Bouldin** — average similarity between each cluster and its most similar one. Lower
  is better.
- **Gap statistic** — compare $\log(\text{inertia})$ against what you would get on uniform
  random data of the same shape. Uniquely, it can return $k=1$.

Run all five on data whose answer we know.

In [ ]:
print(f"data: {len(np.unique(y6))} true clusters, {len(X6)} points\n")
print(f"{'k':>3} {'inertia':>11} {'silhouette':>12} {'Calinski-H':>12} {'Davies-B':>11}")
print("-" * 52)
ks = list(range(2, 11))
rows = {}
for k in ks:
    lab = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X6)
    rows[k] = (lab.inertia_, silhouette_score(X6, lab.labels_),
               calinski_harabasz_score(X6, lab.labels_),
               davies_bouldin_score(X6, lab.labels_))
    print(f"{k:>3} {rows[k][0]:>11.1f} {rows[k][1]:>12.4f} {rows[k][2]:>12.1f} "
          f"{rows[k][3]:>11.4f}")

inert = np.array([rows[k][0] for k in ks])
elbow_k = ks[int(np.argmax(np.diff(inert, 2))) + 1]
print(f"\n  elbow (largest second difference) picks k = {elbow_k}")
print(f"  silhouette picks                     k = {max(rows, key=lambda k: rows[k][1])}")
print(f"  Calinski-Harabasz picks              k = {max(rows, key=lambda k: rows[k][2])}")
print(f"  Davies-Bouldin picks                 k = {min(rows, key=lambda k: rows[k][3])}")
print(f"  THE TRUTH IS                         k = {len(np.unique(y6))}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(ks, inert, marker="o"); axes[0].set(xlabel="k", ylabel="inertia", title="Elbow")
axes[1].plot(ks, [rows[k][1] for k in ks], marker="o", color="seagreen")
axes[1].set(xlabel="k", ylabel="silhouette", title="Silhouette (higher better)")
axes[2].plot(ks, [rows[k][2] for k in ks], marker="o", color="crimson")
axes[2].set(xlabel="k", ylabel="Calinski-Harabasz", title="Calinski-Harabasz (higher better)")
for ax in axes:
    ax.axvline(6, color="gray", ls=":", lw=1)
    ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print()
print("Three of the four are WRONG, and they disagree with each other. The dotted line is")
print("the truth.")
print()
print("Why silhouette prefers k=3: it rewards well-separated, compact clusters, and merging")
print("the six blobs into three tight super-clusters scores better on that criterion than")
print("recovering the six. Silhouette is not measuring correctness - it is measuring a")
print("geometric property that only sometimes coincides with it.")
print()
print("Treat all of these as evidence, never as an answer. If domain knowledge says the")
print("answer should be 6, a metric preferring 3 is not a reason to change your mind.")

In [ ]:
def gap_statistic(X, k_max=8, n_ref=10, seed=RANDOM_STATE):
    """Tibshirani, Walther & Hastie (2001).

    Compare log(inertia) on the real data against uniform reference data spanning the
    same box. Structure shows up as the real data doing much better than uniform.
    """
    rng = np.random.default_rng(seed)
    lo, hi = X.min(axis=0), X.max(axis=0)
    gaps, sks = [], []
    for k in range(1, k_max + 1):
        wk = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X).inertia_
        refs = np.array([
            np.log(KMeans(n_clusters=k, n_init=10, random_state=seed)
                   .fit(rng.uniform(lo, hi, size=X.shape)).inertia_)
            for _ in range(n_ref)])
        gaps.append(refs.mean() - np.log(wk))
        sks.append(refs.std() * np.sqrt(1 + 1 / n_ref))
    return np.array(gaps), np.array(sks)


def gap_choose_k(gaps, sks):
    """Smallest k with gap(k) >= gap(k+1) - s(k+1)."""
    for i in range(len(gaps) - 1):
        if gaps[i] >= gaps[i + 1] - sks[i + 1]:
            return i + 1
    return len(gaps)


gaps, sks = gap_statistic(X6, k_max=8, n_ref=10)
print("gap statistic on the 6-blob data:\n")
print(f"  {'k':>3} {'gap':>9} {'s_k':>8}")
print("  " + "-" * 22)
choice = gap_choose_k(gaps, sks)
for i, (g, s) in enumerate(zip(gaps, sks), start=1):
    print(f"  {i:>3} {g:>9.4f} {s:>8.4f}" + ("   <-- chosen" if i == choice else ""))
print(f"\n  gap statistic picks k = {choice}   (truth: {len(np.unique(y6))})")
print()
print("The gap statistic gets it right where three of the other four did not, and it is the")
print("only one of the five that can even express 'k=1, there are no clusters' - which is")
print("what 1.8 is about.")
print()
print("The cost is compute: it refits k-means on n_ref synthetic datasets for every k.")

## 1.8 Clusters from pure noise

The failure mode that matters most in practice. K-means minimises WCSS, and **WCSS can always
be reduced by adding centres** — whether or not the data contains any groups at all.

So the algorithm cannot decline. Feed it uniform random points and it will return a tidy
partition with plausible-looking scores.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_noise = rng.uniform(0, 1, size=(500, 2))          # no structure whatsoever

km_noise = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit(X_noise)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
axes[0].scatter(X_noise[:, 0], X_noise[:, 1], s=8, c="gray", alpha=0.6)
axes[0].set_title("500 uniform random points", fontsize=10)
axes[1].scatter(X_noise[:, 0], X_noise[:, 1], c=km_noise.labels_, cmap="Set1", s=8, alpha=0.8)
axes[1].scatter(km_noise.cluster_centers_[:, 0], km_noise.cluster_centers_[:, 1],
                c="black", marker="X", s=110)
axes[1].set_title("k-means, k=5: five tidy 'clusters'", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

print(f"{'k':>3} {'inertia':>11} {'silhouette':>12} {'Calinski-H':>12}")
print("-" * 40)
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_noise)
    print(f"{k:>3} {km.inertia_:>11.4f} {silhouette_score(X_noise, km.labels_):>12.4f} "
          f"{calinski_harabasz_score(X_noise, km.labels_):>12.1f}")

sil_noise = silhouette_score(X_noise, KMeans(n_clusters=6, n_init=10,
                                             random_state=RANDOM_STATE).fit_predict(X_noise))
sil_real = silhouette_score(X6, KMeans(n_clusters=6, n_init=10,
                                       random_state=RANDOM_STATE).fit_predict(X6))
print(f"\nsilhouette at k=6, real blobs : {sil_real:.4f}")
print(f"silhouette at k=6, pure noise : {sil_noise:.4f}")
print()
print("The noise scores lower - but it scores around 0.4, and a silhouette of 0.4 is")
print("routinely reported as 'reasonable structure'. Nothing in that number announces that")
print("the data is uniform.")

In [ ]:
# The gap statistic is the one that catches it - but check that across samples, not once.
picks = []
for seed in range(8):
    r = np.random.default_rng(1000 + seed)
    g, s = gap_statistic(r.uniform(0, 1, size=(500, 2)), k_max=6, n_ref=8)
    picks.append(gap_choose_k(g, s))
print(f"gap statistic's chosen k on 8 independent uniform-noise samples: {picks}")
print(f"fraction correctly answering k=1: {np.mean(np.array(picks) == 1):.0%}")

g_real, _ = gap_statistic(X6, k_max=6, n_ref=8)
g_noise, _ = gap_statistic(rng.uniform(0, 1, size=(500, 2)), k_max=6, n_ref=8)
print(f"\nlargest gap VALUE, 6 real blobs : {g_real.max():.4f}")
print(f"largest gap VALUE, uniform noise: {g_noise.max():.4f}")
print()
print("Two things to take away.")
print()
print("1. The gap statistic answers 'there are no clusters here' correctly and repeatedly,")
print("   which none of the other four can do. If you are ever unsure whether your data has")
print("   cluster structure at all, this is the test to run.")
print()
print("2. Read the VALUE, not just the argmax. The gap on real structure is an order of")
print("   magnitude larger than on noise. A selection rule can misfire on a single sample")
print("   while the magnitude still tells you plainly what you are looking at.")
print()
print("And the general lesson, which outlives k-means: an algorithm that cannot return")
print("'nothing here' will always return something. Before interpreting any clustering,")
print("establish that there is structure to interpret.")

---
# Part 2 - Worked example: clustering digits with the labels hidden

**The task.** 1,797 handwritten digits, 64 pixels each. Pretend nobody told you what they are.
Can clustering recover the ten digits?

**Why this dataset.** It is the rare clustering problem with a *known* answer, so we can measure
what the internal metrics could not have told us. That is the whole point: every number the
metrics give you in §2.3 is available in real work, and every number in §2.4 is not.

## 2.1 Set up honestly

The labels exist, so the discipline is to use them **only at the end**, as an external
validation — never to choose $k$, the preprocessing, or the method.

In [ ]:
X_dig, y_dig = DIGITS.data, DIGITS.target
print(f"{X_dig.shape[0]} images, {X_dig.shape[1]} pixels, values {X_dig.min():.0f}-{X_dig.max():.0f}")
print(f"(the labels exist but we will not look until 2.4)")

X_scaled = StandardScaler().fit_transform(X_dig)
X_pca = PCA(n_components=20, random_state=RANDOM_STATE).fit_transform(X_dig)
print(f"\nrepresentations to try:")
print(f"  raw pixels        {X_dig.shape}")
print(f"  scaled pixels     {X_scaled.shape}")
print(f"  PCA(20) of pixels {X_pca.shape}  "
      f"({PCA(n_components=20, random_state=RANDOM_STATE).fit(X_dig).explained_variance_ratio_.sum():.1%} "
      f"of variance)")
print()
print("PCA before clustering is the standard move (NB-09), for two reasons: it denoises, and")
print("distance-based methods degrade in high dimensions (NB-07 section 1.5).")

## 2.2 Choosing k without looking at the labels

In [ ]:
print("Everything here is computable WITHOUT labels - this is what you would really have.\n")
print(f"{'k':>3} {'inertia':>12} {'silhouette':>12} {'Calinski-H':>12} {'Davies-B':>10}")
print("-" * 54)
ks_d = [4, 6, 8, 10, 12, 15, 20]
res_d = {}
for k in ks_d:
    lab = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_pca)
    res_d[k] = (lab.inertia_, silhouette_score(X_pca, lab.labels_),
                calinski_harabasz_score(X_pca, lab.labels_),
                davies_bouldin_score(X_pca, lab.labels_))
    print(f"{k:>3} {res_d[k][0]:>12.1f} {res_d[k][1]:>12.4f} {res_d[k][2]:>12.1f} "
          f"{res_d[k][3]:>10.4f}")

print(f"\n  silhouette picks        k = {max(res_d, key=lambda k: res_d[k][1])}")
print(f"  Calinski-Harabasz picks k = {max(res_d, key=lambda k: res_d[k][2])}")
print(f"  Davies-Bouldin picks    k = {min(res_d, key=lambda k: res_d[k][3])}")
print()
print("We happen to know the answer is 10. Note whether any of these would have told you.")
print()
print("This is the realistic situation: the metrics point in different directions, and the")
print("thing that actually settles it is DOMAIN KNOWLEDGE - 'there are ten digits' - which")
print("no amount of unsupervised analysis could have supplied.")

In [ ]:
# The silhouette plot: more informative than the single number.
km10 = KMeans(n_clusters=10, n_init=10, random_state=RANDOM_STATE).fit(X_pca)
sil_vals = silhouette_samples(X_pca, km10.labels_)

fig, ax = plt.subplots(figsize=(7, 4.5))
y_lower = 0
for i in range(10):
    vals = np.sort(sil_vals[km10.labels_ == i])
    ax.fill_betweenx(np.arange(y_lower, y_lower + len(vals)), 0, vals, alpha=0.8)
    ax.text(-0.05, y_lower + len(vals) / 2, str(i), fontsize=8, va="center")
    y_lower += len(vals) + 10
ax.axvline(sil_vals.mean(), color="crimson", ls="--", lw=1,
           label=f"mean {sil_vals.mean():.3f}")
ax.set(xlabel="silhouette coefficient", ylabel="cluster", title="Silhouette plot, k=10")
ax.legend(fontsize=8); ax.set_yticks([])
fig.tight_layout(); plt.show()

neg = (sil_vals < 0).sum()
print(f"points with NEGATIVE silhouette: {neg} of {len(sil_vals)} ({neg/len(sil_vals):.1%})")
print("A negative value means the point is on average closer to a DIFFERENT cluster than to")
print("its own - it is assigned to the wrong one by the metric's own standard.")
print()
print("Read the shape, not the mean. Clusters with a short, thin wedge are weak; clusters")
print("with many negative values are not really separate. The single averaged number hides")
print("all of that, which is why a silhouette plot beats a silhouette score.")

## 2.3 Which representation?

Still no labels. All we can compare is internal metrics — which, as §2.2 just showed, may not
point at the right answer.

In [ ]:
print(f"{'representation':<28} {'inertia':>12} {'silhouette':>12} {'time (s)':>10}")
print("-" * 66)
for name, data in [("raw pixels", X_dig), ("scaled pixels", X_scaled), ("PCA(20)", X_pca)]:
    t0 = time.time()
    km = KMeans(n_clusters=10, n_init=10, random_state=RANDOM_STATE).fit(data)
    t = time.time() - t0
    print(f"{name:<28} {km.inertia_:>12.1f} "
          f"{silhouette_score(data, km.labels_):>12.4f} {t:>10.2f}")

print()
print("Inertia is NOT comparable across representations - it is measured in different units")
print("in each. The scaled-pixel inertia is smaller than the others by more than an order of")
print("magnitude purely because standardising shrank the numbers, not because the clustering")
print("is better. Only the silhouette column can be read down the page, and even that is")
print("affected by dimensionality.")
print()
print("This is a real limitation, not a presentational one. Without labels you have very")
print("little basis for choosing a representation. 2.4 shows what the right answer was.")

## 2.4 Now look at the labels

Everything below uses `y_dig`, and **none of it was available for any decision above**. This is
the part of clustering work you almost never get in reality.

In [ ]:
print(f"{'method':<36} {'ARI':>8} {'NMI':>8} {'silhouette':>12} {'time (s)':>10}")
print("-" * 78)
outcomes = {}
for name, model, data in [
    ("k-means (k=10), raw pixels", KMeans(n_clusters=10, n_init=10,
                                          random_state=RANDOM_STATE), X_dig),
    ("k-means (k=10), scaled pixels", KMeans(n_clusters=10, n_init=10,
                                             random_state=RANDOM_STATE), X_scaled),
    ("k-means (k=10), PCA(20)", KMeans(n_clusters=10, n_init=10,
                                       random_state=RANDOM_STATE), X_pca),
    ("GMM (10 components), PCA(20)", GaussianMixture(n_components=10, n_init=3,
                                                     random_state=RANDOM_STATE), X_pca),
    ("Ward hierarchical (k=10), raw", AgglomerativeClustering(n_clusters=10), X_dig),
]:
    t0 = time.time()
    lab = model.fit_predict(data)
    t = time.time() - t0
    outcomes[name] = lab
    print(f"{name:<36} {adjusted_rand_score(y_dig, lab):>8.4f} "
          f"{normalized_mutual_info_score(y_dig, lab):>8.4f} "
          f"{silhouette_score(data, lab):>12.4f} {t:>10.2f}")

print()
print("Three results worth sitting with.")
print()
print("1. SCALING HURTS here - the opposite of 1.6's wine result. Digits pixels already")
print("   share one 0-16 scale, so standardising them amplifies the near-constant border")
print("   pixels into full-weight features. Same argument as NB-09 section 1.5: scale when")
print("   units differ, not by reflex.")
print()
print("2. Ward hierarchical beats k-means clearly, and it was not the method any internal")
print("   metric would have chosen.")
print()
print("3. The silhouette column does NOT rank the methods the way ARI does. The metric you")
print("   would have had available disagrees with the metric that measures correctness.")

In [ ]:
# Which digits get merged? The confusion matrix, read the clustering way.
best_lab = outcomes["k-means (k=10), PCA(20)"]
cm = confusion_matrix(y_dig, best_lab)
print("rows = true digit, columns = cluster id (ids are arbitrary)\n")
print("     " + " ".join(f"{c:>4}" for c in range(10)))
for d in range(10):
    print(f"  {d}: " + " ".join(f"{v:>4}" for v in cm[d]))

print("\nfor each true digit, the share captured by its single largest cluster:")
for d in range(10):
    print(f"  digit {d}: {cm[d].max() / cm[d].sum():>6.1%}", end="")
    if d % 2 == 1:
        print()

purity = cm.max(axis=0).sum() / cm.sum()
print(f"\noverall cluster purity: {purity:.1%}")
print()
worst = np.argsort([cm[d].max() / cm[d].sum() for d in range(10)])[:3]
best = np.argsort([cm[d].max() / cm[d].sum() for d in range(10)])[-3:]
print(f"most fragmented digits: {sorted(worst.tolist())}   "
      f"purest: {sorted(best.tolist())}")
print("The fragmented ones are those a human also finds ambiguous at 8x8 resolution - they")
print("share strokes with other digits and get split across clusters - while 0, 6 and 7 are")
print("nearly pure.")
print()
print("Note what this table is NOT: it is not an accuracy score. Clusters have no names, so")
print("there is no 'predicted digit' - only groups that happen to align with digits. Purity")
print("as computed here also cheats slightly, because it assigns each cluster its best-")
print("matching digit AFTER seeing the labels.")

---
# Part 3 - The other three methods

K-means is one point in a space of choices. Three others cover most of the rest, and each
relaxes a different k-means assumption.

## 3.1 Hierarchical clustering

Build a tree instead of a partition. **Agglomerative** clustering starts with every point in
its own cluster and repeatedly merges the two closest, recording the whole history as a
**dendrogram**. Cut the tree at any height to get a clustering.

You do not have to choose $k$ in advance — but you do have to choose **linkage**, which
defines the distance between two *clusters*, and that choice matters more than $k$ does.

In [ ]:
X_hb, y_hb = make_blobs(n_samples=400, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
X_hm, y_hm = make_moons(n_samples=400, noise=0.06, random_state=RANDOM_STATE)

print(f"{'linkage':<12} {'what it merges on':<44} {'blobs ARI':>10} {'moons ARI':>10}")
print("-" * 80)
descriptions = {
    "ward": "minimises the increase in within-cluster variance",
    "complete": "distance between the two FARTHEST members",
    "average": "mean distance between all cross-pairs",
    "single": "distance between the two CLOSEST members",
}
for link in ["ward", "complete", "average", "single"]:
    a_b = adjusted_rand_score(y_hb, AgglomerativeClustering(n_clusters=3,
                                                            linkage=link).fit_predict(X_hb))
    a_m = adjusted_rand_score(y_hm, AgglomerativeClustering(n_clusters=2,
                                                            linkage=link).fit_predict(X_hm))
    print(f"{link:<12} {descriptions[link]:<44} {a_b:>10.4f} {a_m:>10.4f}")

print()
print("Every linkage solves the blobs. Only 'single' solves the moons, because it merges on")
print("the nearest pair and so can follow a chain of points around a curve.")
print()
print("Before adopting single linkage, see what that same chaining does when the data is")
print("slightly less clean.")

In [ ]:
# Single linkage's chaining, made concrete.
rng = np.random.default_rng(RANDOM_STATE)
bridge = np.column_stack([np.linspace(X_hm[:, 0].min(), X_hm[:, 0].max(), 12),
                          np.full(12, 0.25)])
X_br = np.vstack([X_hm, bridge])
y_br = np.concatenate([y_hm, np.full(12, 0)])

print(f"add 12 points forming a thin bridge between the moons ({len(X_br)} points total):\n")
print(f"  {'linkage':<12} {'ARI before':>12} {'ARI after':>12}")
print("  " + "-" * 38)
for link in ["ward", "average", "single"]:
    before = adjusted_rand_score(y_hm, AgglomerativeClustering(n_clusters=2,
                                                               linkage=link).fit_predict(X_hm))
    after = adjusted_rand_score(y_br, AgglomerativeClustering(n_clusters=2,
                                                              linkage=link).fit_predict(X_br))
    print(f"  {link:<12} {before:>12.4f} {after:>12.4f}")

print()
print("Single linkage goes from a perfect 1.0 to essentially zero. Twelve points out of 412")
print("- 3% of the data - destroy it completely, because the chain now runs from one moon")
print("to the other.")
print()
print("That is the CHAINING EFFECT, and it is why single linkage is rarely used despite")
print("being the only linkage that handles non-convex shapes. Ward is the sensible default.")

In [ ]:
# The dendrogram: the output that no other method gives you.
X_small, y_small = make_blobs(n_samples=40, centers=3, cluster_std=1.0,
                              random_state=RANDOM_STATE)
Z = linkage(X_small, method="ward")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
dendrogram(Z, ax=axes[0], color_threshold=0)
axes[0].set(title="Dendrogram (Ward linkage)", xlabel="point", ylabel="merge distance")
axes[0].tick_params(axis="x", labelsize=5)

dendrogram(Z, ax=axes[1], color_threshold=Z[-2, 2])
axes[1].axhline(Z[-2, 2], color="crimson", ls="--", lw=1)
axes[1].set(title="Cut here -> 3 clusters", xlabel="point")
axes[1].tick_params(axis="x", labelsize=5)
fig.tight_layout(); plt.show()

merge_heights = Z[:, 2]
print("the largest jumps in merge distance (a heuristic for where to cut):")
jumps = np.diff(merge_heights)
for i in np.argsort(jumps)[-3:][::-1]:
    print(f"  merging at height {merge_heights[i]:.2f} -> {merge_heights[i+1]:.2f} "
          f"leaves {len(X_small) - i - 1} clusters")

print()
print("The dendrogram is the real advantage of hierarchical clustering: it shows the whole")
print("nesting structure at once, so you can see whether a clean k even exists. A tree with")
print("one obvious tall branch says 'there are 2 groups'; a tree that merges smoothly all")
print("the way up says the data has no natural k at all.")
print()
print("The cost is O(n^2) memory and roughly O(n^3) time for the naive algorithm - it does")
print("not scale past tens of thousands of points, which is why k-means survives.")

## 3.2 DBSCAN

**Density-based** clustering. A point is a *core point* if at least `min_samples` points lie
within distance `eps` of it; clusters are connected components of core points, and anything
left over is labelled **noise** (`-1`).

Two genuine advantages: it finds arbitrarily shaped clusters, and it is the only method here
that can say "this point belongs to nothing". The catch is that it trades the problem of
choosing $k$ for the problem of choosing `eps`.

In [ ]:
X_dm = StandardScaler().fit_transform(X_hm)
print("two moons, 400 points, scaled\n")
print(f"  {'eps':>6} {'clusters':>10} {'noise points':>14} {'ARI':>9}")
print("  " + "-" * 44)
for eps in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.80]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X_dm)
    print(f"  {eps:>6.2f} {len(set(lab) - {-1}):>10} {(lab == -1).sum():>14} "
          f"{adjusted_rand_score(y_hm, lab):>9.4f}")

print()
print("'DBSCAN does not make you choose k' is true and misleading. You choose eps instead,")
print("and over this range the answer runs from 'almost everything is noise' through 26")
print("clusters to a single one - with no principled way to pick without labels.")
print()
print("The standard heuristic - plot the distance to each point's k-th nearest neighbour,")
print("sorted, and look for the knee - is the elbow method again, with the same weakness.")

In [ ]:
# Where DBSCAN genuinely wins, and where it genuinely loses.
X_ci, y_ci = make_circles(n_samples=500, factor=0.4, noise=0.06, random_state=RANDOM_STATE)
X_ci = StandardScaler().fit_transform(X_ci)

print(f"{'dataset':<28} {'k-means ARI':>13} {'DBSCAN ARI':>12} {'GMM ARI':>10}")
print("-" * 66)
for name, Xd, yd, k, eps in [("two moons", X_dm, y_hm, 2, 0.30),
                             ("concentric circles", X_ci, y_ci, 2, 0.30)]:
    a_km = adjusted_rand_score(yd, KMeans(n_clusters=k, n_init=10,
                                          random_state=RANDOM_STATE).fit_predict(Xd))
    a_db = adjusted_rand_score(yd, DBSCAN(eps=eps, min_samples=5).fit_predict(Xd))
    a_gm = adjusted_rand_score(yd, GaussianMixture(n_components=k, n_init=5,
                                                   random_state=RANDOM_STATE).fit_predict(Xd))
    print(f"{name:<28} {a_km:>13.4f} {a_db:>12.4f} {a_gm:>10.4f}")

print("\nNow the same method on the digits from Part 2 (64 dimensions):\n")
print(f"  {'eps':>6} {'clusters':>10} {'noise points':>14} {'ARI':>9}")
print("  " + "-" * 44)
for eps in [5, 12, 20, 25, 30]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X_dig)
    print(f"  {eps:>6} {len(set(lab) - {-1}):>10} {(lab == -1).sum():>14} "
          f"{adjusted_rand_score(y_dig, lab):>9.4f}")

print()
print("DBSCAN wins outright on curved 2-D shapes - k-means and GMM both fail there and it")
print("does not.")
print()
print("On 64-dimensional digits it never works well: either almost everything is noise, or")
print("everything collapses into one cluster, with a narrow band of eps in between that")
print("produces far too many clusters.")
print()
print("The reason is NB-07 section 1.5. DBSCAN is built on a fixed distance threshold, and")
print("in high dimensions all pairwise distances converge - so there is no eps that is")
print("simultaneously small enough to separate clusters and large enough to connect them.")
print("Density-based clustering is a low-dimensional tool.")

## 3.3 Gaussian mixture models

Fit a **probabilistic model**: assume the data was generated by $k$ Gaussians with unknown
means, covariances and weights, and fit them by expectation-maximisation.

Two things follow that k-means cannot do:

- **Full covariance per component**, so clusters can be elongated and differently shaped —
  which is why GMM held up in §1.5 where k-means collapsed.
- **Soft assignment**: every point gets a probability of belonging to each cluster.

In fact **k-means is a limiting case of a GMM** — spherical covariances of equal, vanishing
variance, with hard assignment.

In [ ]:
print("Soft assignment only carries information when clusters actually overlap:\n")
print(f"  {'cluster_std':>12} {'assigned >0.99':>16} {'ambiguous <0.80':>17} "
      f"{'GMM ARI':>9} {'k-means ARI':>13}")
print("  " + "-" * 74)
for std in [1.0, 2.0, 3.0, 4.0]:
    Xg, yg = make_blobs(n_samples=600, centers=4, cluster_std=std,
                        random_state=RANDOM_STATE)
    gm = GaussianMixture(n_components=4, n_init=5, random_state=RANDOM_STATE).fit(Xg)
    conf = gm.predict_proba(Xg).max(axis=1)
    a_gm = adjusted_rand_score(yg, gm.predict(Xg))
    a_km = adjusted_rand_score(yg, KMeans(n_clusters=4, n_init=10,
                                          random_state=RANDOM_STATE).fit_predict(Xg))
    print(f"  {std:>12.1f} {(conf > 0.99).mean():>15.1%} {(conf < 0.80).mean():>17.1%} "
          f"{a_gm:>9.4f} {a_km:>13.4f}")

print()
print("At cluster_std=1.0 almost every point is assigned with certainty and the soft output")
print("tells you nothing k-means did not. By std=4.0, a third of points are genuinely")
print("ambiguous - and being able to SEE that is the advantage.")
print()
print("Note also that GMM does not beat k-means on ARI in these rows. The soft assignment is")
print("extra information, not automatically a better partition.")

In [ ]:
# BIC: a principled-looking way to choose k, with the same blind spot.
Xg, yg = make_blobs(n_samples=600, centers=4, cluster_std=1.4, random_state=RANDOM_STATE)
print("BIC across k (lower is better). Truth: 4 clusters.\n")
print(f"  {'k':>3} {'BIC':>12} {'AIC':>12}")
print("  " + "-" * 30)
bic = {}
for k in range(1, 9):
    g = GaussianMixture(n_components=k, n_init=5, random_state=RANDOM_STATE).fit(Xg)
    bic[k] = g.bic(Xg)
    print(f"  {k:>3} {g.bic(Xg):>12.1f} {g.aic(Xg):>12.1f}")
print(f"\n  BIC picks k = {min(bic, key=bic.get)}  <- correct")

rng2 = np.random.default_rng(RANDOM_STATE)
X_n2 = rng2.uniform(0, 1, size=(500, 2))
bic_n = {k: GaussianMixture(n_components=k, n_init=5,
                            random_state=RANDOM_STATE).fit(X_n2).bic(X_n2)
         for k in range(1, 7)}
print(f"  BIC on uniform NOISE picks k = {min(bic_n, key=bic_n.get)}  (truth is 1)")

print()
print("BIC gets the real data right, and it has a genuine advantage over silhouette: it is")
print("a model-selection criterion with a likelihood behind it, and it CAN evaluate k=1.")
print()
print("But it still chooses several components on uniform noise, because a mixture of")
print("Gaussians genuinely does fit a uniform square better than one Gaussian does. BIC is")
print("answering 'which model fits best?', not 'are there clusters?' - and those are")
print("different questions. Only the gap statistic (1.8) asks the second one.")

## 3.4 Choosing between them

Nothing here is a default. Match the method to what you know about the problem.

| | k-means | Hierarchical | DBSCAN | GMM |
|---|---|---|---|---|
| **Must choose** | k | linkage (+ where to cut) | eps, min_samples | k, covariance type |
| **Cluster shape** | round | depends on linkage | any | ellipsoidal |
| **Unequal sizes** | **fails badly** (§1.5) | **depends on linkage** — Ward fails too | tolerant | tolerant |
| **Handles noise** | no | no | **yes** | no (but gives low probabilities) |
| **Assignment** | hard | hard | hard | **soft** |
| **Scales to n** | **excellent** | poor — $O(n^2)$ memory | moderate | good |
| **High dimensions** | tolerable | tolerable | **poor** (§3.2) | tolerable |
| **Deterministic** | no (§1.3) | **yes** | **yes** | no |

**Start with k-means** because it is fast and you will need a baseline. Reach for
**hierarchical** when you want to see the structure rather than commit to a k, **DBSCAN** when
clusters are irregular in low dimensions and outliers matter, and **GMM** when clusters overlap
and you want a probability rather than a verdict.

⚠️ **Note the "depends on linkage" cell.** Ward linkage merges whichever pair least increases
within-cluster variance — which is k-means' objective, reached by a different route. So Ward
inherits k-means' bias against small clusters, and §3.4's table measures it scoring *worse*
than k-means on unequal sizes while **average** linkage handles them comfortably. "Use
hierarchical instead" is not a fix for §1.5's failure unless you also change the linkage.

In [ ]:
# Everything, on everything. The honest summary table.
datasets = [
    ("blobs (round, equal)", *make_blobs(n_samples=500, centers=3, cluster_std=1.0,
                                         random_state=RANDOM_STATE)),
    ("unequal sizes 550/40/10", *make_blobs(n_samples=[550, 40, 10], centers=None,
                                            n_features=2, cluster_std=1.5,
                                            center_box=(-6, 6), random_state=RANDOM_STATE)),
    ("two moons", *make_moons(n_samples=500, noise=0.06, random_state=RANDOM_STATE)),
    ("concentric circles", *make_circles(n_samples=500, factor=0.4, noise=0.06,
                                         random_state=RANDOM_STATE)),
]

print(f"{'dataset':<26} {'k-means':>9} {'Ward':>8} {'DBSCAN':>9} {'GMM':>8}")
print("-" * 64)
for name, Xd, yd in datasets:
    Xd = StandardScaler().fit_transform(Xd)
    k = len(np.unique(yd))
    a_km = adjusted_rand_score(yd, KMeans(n_clusters=k, n_init=10,
                                          random_state=RANDOM_STATE).fit_predict(Xd))
    a_hc = adjusted_rand_score(yd, AgglomerativeClustering(n_clusters=k).fit_predict(Xd))
    best_db = max(adjusted_rand_score(yd, DBSCAN(eps=e, min_samples=5).fit_predict(Xd))
                  for e in np.linspace(0.1, 0.9, 17))
    a_gm = adjusted_rand_score(yd, GaussianMixture(n_components=k, n_init=5,
                                                   random_state=RANDOM_STATE).fit_predict(Xd))
    print(f"{name:<26} {a_km:>9.4f} {a_hc:>8.4f} {best_db:>9.4f} {a_gm:>8.4f}")

print()
print("Two caveats that make this table less useful than it looks, and both matter more")
print("than the numbers in it:")
print()
print("  1. DBSCAN's eps was tuned by trying 17 values and keeping the best ARI - which")
print("     requires the labels. Every other column used a fixed setting. That is not a")
print("     fair comparison, and it is exactly the comparison people publish.")
print()
print("  2. ARI requires labels at all. In genuine clustering work you cannot build this")
print("     table, which is why 2.2 and 2.3 - the label-free sections - are the ones that")
print("     resemble real practice.")
print()
print("If you take one thing from Part 3: the method matters less than knowing what shape")
print("of structure you are looking for, and no metric can tell you that.")

---
# Part 4 - Tough questions

---

### Q1. Explain k-means in two sentences, then say what it is actually optimising.

<details><summary>Answer</summary>

**Two sentences:** assign every point to the nearest of $k$ centres, then move each centre to
the mean of the points assigned to it. Repeat until nothing moves.

**What it optimises:** the **within-cluster sum of squares**,

$$ \text{WCSS} = \sum_{j=1}^{k}\sum_{x \in C_j}\lVert x - \mu_j\rVert^2 $$

Both steps decrease it — the assign step by definition of "nearest", the update step because
the mean is the point minimising summed squared distance. Since there are finitely many
possible assignments and the objective never increases, it must terminate.

**Three things that follow from the objective**, and they are the whole notebook:

1. **Squared Euclidean distance** ⇒ round clusters, sensitivity to scale (§1.6) and to outliers.
2. **A sum over points** ⇒ big clusters dominate. §1.5 measures k-means collapsing from ARI
   0.93 to **0.03** on sizes 580/15/5, because splitting the big cluster reduces WCSS more than
   isolating the small one does.
3. **WCSS always falls as $k$ rises** ⇒ the objective can never tell you $k$, and can never say
   "there are no clusters" (§1.8).

Finding the global WCSS optimum is NP-hard. Lloyd's algorithm is a heuristic that finds a local
one (§1.3).

</details>

---

### Q2. Why does sklearn default to `n_init=10`, and what happens if you set it to 1?

<details><summary>Answer</summary>

**Because Lloyd's algorithm finds a local optimum that depends entirely on where it started.**

§1.3 runs it 30 times on the same data with `n_init=1` and random initialisation:

- **21 distinct solutions** out of 30 runs
- worst inertia **1.71×** the best
- only **8 of 30** runs found the best solution

With `n_init=1` you get one draw from that distribution and no way to know which. `n_init=10`
runs the whole algorithm ten times and keeps the lowest inertia, which is cheap insurance —
k-means is fast, and this is the single highest-value default in the estimator.

**k-means++ helps but does not fix it** (§1.4). It seeds centres far apart, and the guarantee is
that expected WCSS is within $O(\log k)$ of optimal *before* Lloyd's runs. §1.4 measures the
spread narrowing, and notes it matters more as $k$ grows: at $k=5$ random and k-means++ were
indistinguishable; by $k=20$ the gap was large.

**Practical rule:** never lower `n_init` to save time. If k-means is too slow, use
`MiniBatchKMeans`, subsample, or reduce dimensions first — do not economise on restarts.

</details>

---

### Q3. How do you choose k?

<details><summary>Answer</summary>

Honestly: **you mostly cannot, from the data alone.** §1.7 runs the standard methods on data
with a known answer of **6**:

| method | picks |
|---|---|
| elbow (largest second difference) | 3 |
| silhouette | 3 |
| Davies–Bouldin | 4 |
| Calinski–Harabasz | **6** ✓ |
| gap statistic | **6** ✓ |

Three of five are wrong and they disagree with each other. That is not a badly chosen example —
it is the normal situation.

**What each is really measuring:** silhouette rewards compact, well-separated clusters, so it
prefers merging six real blobs into three tighter super-clusters. It is not measuring
correctness; it is measuring a geometric property that only sometimes coincides with it.

**What to actually do:**

1. **Use domain knowledge first.** "There are ten digits", "marketing wants four segments",
   "the assay has three known subtypes". This beats every metric, and §2.2 shows it being the
   only thing that identifies $k=10$ on digits.
2. **Compute several metrics and look for agreement.** Disagreement is information: it means the
   structure is not clean.
3. **Use the gap statistic if you need to know whether clusters exist at all** — it is the only
   one of the five that can answer $k=1$ (§1.8).
4. **Check stability** (Q11). A $k$ whose clusters change completely on a resample is not a real
   $k$.
5. **Validate externally.** Do the clusters predict something you care about? That is the only
   test that means anything.

</details>

---

### Q4. Your silhouette score is 0.42. Is that good?

<details><summary>Answer</summary>

**Unanswerable as asked — and §1.8 shows why that is the point.**

Running k-means on **500 uniformly random points with no structure whatsoever** produces
silhouette scores of **0.38 to 0.42** across $k=2$ to $8$. A score of 0.42 is exactly what pure
noise gives you.

For comparison, six genuinely separated blobs score **0.57** at the correct $k$. So the real
structure does score higher — but 0.42 is not near zero, and the rules of thumb that circulate
("above 0.25 is reasonable structure", "above 0.5 is strong") would have called the noise a
success.

**What to do instead:**

- **Compare against a null.** That is precisely what the gap statistic does (§1.8), and on 8
  independent noise samples it correctly answered $k=1$ **every time**. If you only remember one
  technique from this notebook, this is it.
- **Plot the silhouette, do not average it** (§2.2). The per-cluster shape shows which clusters
  are weak and how many points have *negative* values — meaning they are closer to another
  cluster than their own. The single number hides all of it.
- **Ask what the number is for.** Silhouette compares clusterings *of the same data*. It is
  reasonable for choosing between $k=4$ and $k=5$; it is not a certificate that clusters exist.

**And never compare silhouette across different representations** — §2.3 shows raw pixels,
scaled pixels and PCA(20) producing scores that are not on a common footing, because the
distances they measure live in different spaces.

</details>

---

### Q5. When does k-means fail, and which failure actually matters?

<details><summary>Answer</summary>

The textbook list is: non-spherical clusters, unequal variances, unequal sizes, non-convex
shapes. §1.5 measures them, and the ranking is not what you would guess.

**Anisotropy barely matters when clusters are separated.** With `cluster_std=0.6`, k-means
scores a **perfect 1.0** on sheared clusters — the assumption is violated and it costs nothing.
Only at `cluster_std=1.5`, when clusters start to compete for points, does the same shear cost
about 0.39 of ARI.

**Unequal sizes is the failure that matters.** At fixed separation and spread:

| cluster sizes | k-means ARI | GMM ARI |
|---|---|---|
| 200 / 200 / 200 | 0.93 | 0.93 |
| 550 / 40 / 10 | **0.20** | 0.92 |
| 580 / 15 / 5 | **0.03** | 0.84 |

**Why:** WCSS is a *sum*, so a 580-point cluster contributes vastly more total error than a
5-point one. Splitting the big cluster in half reduces the objective more than isolating the
small one. k-means is doing exactly what you asked; the objective does not value small clusters.

**The practical consequence:** if you are clustering to find a *rare* segment — a fraud ring, a
niche customer type, a defect mode — k-means is close to the worst available tool, and it will
not signal that it failed. Use a GMM, DBSCAN (which is built around the idea that some points
belong to nothing), or an anomaly-detection method (NB-15).

**And do not reach for Ward linkage as the fix.** Ward merges whichever pair least increases
within-cluster variance — k-means' objective by another route — so it inherits the same bias.
§3.4 measures Ward scoring *worse* than k-means on unequal sizes, while **average** linkage
handles them comfortably. The failure is in the objective, not in the algorithm that optimises
it.

**Non-convex shapes** (moons, circles) do break k-means outright — §3.4 measures it — but that
failure is at least *visible* in two dimensions. The size failure is invisible.

</details>

---

### Q6. What is the difference between k-means and a Gaussian mixture model?

<details><summary>Answer</summary>

**k-means is a limiting case of a GMM**: spherical covariances of equal, vanishing variance,
with hard assignment.

| | k-means | GMM |
|---|---|---|
| Cluster shape | spherical only | full covariance — any ellipsoid |
| Assignment | hard | **soft** — a probability per cluster |
| Fit by | Lloyd's algorithm | expectation-maximisation |
| Choosing k | no principled criterion | **BIC / AIC** |
| Cost | very fast | slower; more parameters |

**Where the difference pays:**

- **Elongated or differently-shaped clusters.** §1.5: GMM stays at ARI 1.0 on sheared clusters
  where k-means falls to 0.61.
- **Unequal sizes.** §1.5: 0.84 versus 0.03.
- **Overlapping clusters.** §3.3: at `cluster_std=4.0`, **35%** of points have maximum
  probability below 0.80 — genuinely ambiguous, and being able to see that is the advantage.

**Where it does not pay** — and §3.3 measures this too: on well-separated blobs GMM and k-means
give the *same* partition, and at some settings k-means scores slightly higher. Soft assignment
is extra information, not automatically a better clustering.

**The catch:** a full covariance matrix per component is $O(d^2)$ parameters each, so GMM needs
far more data in high dimensions and can be numerically unstable (sklearn has `reg_covar` for
exactly this). If $d$ is large, constrain it — `covariance_type="diag"` or `"spherical"` — and
note that `"spherical"` with hard assignment is k-means again.

</details>

---

### Q7. DBSCAN "does not require you to choose k". Is that an advantage?

<details><summary>Answer</summary>

**It is true, and it is much less of an advantage than it sounds.** You choose `eps` instead.

§3.2, two moons (400 points, scaled):

| eps | clusters found | noise points | ARI |
|---|---|---|---|
| 0.05 | 2 | 387 | 0.00 |
| 0.10 | **26** | 114 | 0.05 |
| 0.20 | 3 | 1 | 0.95 |
| 0.50 | **2** | 0 | **1.00** |
| 0.80 | 1 | 0 | 0.00 |

The answer runs from "almost everything is noise" through 26 clusters to a single one, over a
range you have no labels to choose within. The standard heuristic — sort each point's distance
to its $k$-th nearest neighbour and look for the knee — is the elbow method again, with the same
subjectivity.

**What DBSCAN genuinely gives you:**

- **Arbitrary cluster shapes.** §3.4: it solves moons and concentric circles where k-means and
  GMM both fail completely.
- **An explicit noise label.** It is the only method here that can say a point belongs to
  nothing, which matters when outliers are real.
- **Determinism** (up to border-point ordering), unlike k-means.

**Where it fails badly — and this is under-advertised:** high dimensions. §3.2 runs it on
64-dimensional digits and there is no usable `eps`: either nearly everything is noise, or
everything collapses into one cluster. The reason is NB-07 §1.5 — when all pairwise distances
converge, no single threshold can both separate and connect. **DBSCAN is a low-dimensional
tool**; reduce dimensions first (NB-09) or use something else.

It also struggles with clusters of **varying density**, since one global `eps` cannot suit both
a dense and a sparse region. `HDBSCAN` (in sklearn since 1.3) addresses exactly that and is
usually the better modern choice.

</details>

---

### Q8. How do you evaluate a clustering when you have no labels?

<details><summary>Answer</summary>

This is the central difficulty of the whole subject, and the honest answer is **you evaluate it
badly, so lean on external validation**.

**Internal metrics** (no labels needed) — silhouette, Calinski–Harabasz, Davies–Bouldin. All
measure some version of "compact and well separated". Three limitations, each demonstrated:

1. They **do not identify the right k** — §1.7, three of five wrong on data with a known answer.
2. They **score noise respectably** — §1.8, silhouette 0.42 on uniform random points.
3. They are **biased toward the assumptions of the method that produced the clustering.**
   Silhouette is built on the same compact-and-round notion as k-means, so it will systematically
   favour k-means output over a correct non-convex clustering.

**External metrics** (labels needed) — ARI, NMI, V-measure. These are what §2.4 uses, and they
answer the question properly. If you have labels you are not really doing clustering; you are
validating a clustering, which is a luxury.

**What actually works in practice:**

- **Stability.** Cluster bootstrap resamples and measure whether you get the same partition
  (Q11). Unstable clusters are not findings.
- **A null comparison.** The gap statistic (§1.8) — the only method here that reliably answered
  "no clusters" on noise.
- **Downstream utility.** Do the clusters predict retention, or reduce handling time, or match
  what an expert would draw? That is a real test, and it is external to the clustering.
- **Look at them.** Inspect exemplars from each cluster. §2.4's confusion matrix is exactly this
  done systematically, and it immediately shows which digits merged.

</details>

---

### Q9. Should you scale before clustering?

<details><summary>Answer</summary>

**Usually yes, and this notebook contains a clear counter-example — which is the useful part.**

k-means, hierarchical (with Ward or Euclidean linkage), DBSCAN and GMM all rest on distances, so
a feature with a large numeric range dominates. §1.6 on wine: ARI **0.3711 raw → 0.8975 scaled**,
because the raw clustering is essentially a partition on the single highest-variance chemical.

**But §2.4 measures scaling making digits *worse*** — k-means ARI drops from 0.6669 on raw
pixels to 0.5344 on scaled ones. All 64 pixels already share one 0–16 scale, so standardising
does not fix an incommensurability; it *amplifies* the near-constant border pixels into
full-weight features.

**The rule:**

- **Scale when features have different units.** Almost always for tabular data.
- **Do not scale when features are already commensurable** — pixels, one spectroscopic assay,
  a single sensor array. This is the same conclusion NB-09 §1.5 reaches for PCA, for the same
  reason.

**Related decisions worth making deliberately:**

- **`RobustScaler`** if outliers are distorting the mean and standard deviation — k-means is
  built on squared distance and is very sensitive to them.
- **PCA before clustering** (§2.1) both denoises and shortens distances that stop meaning
  anything in high dimensions (NB-07 §1.5). Whether it helps is empirical — on digits it gave a
  small improvement over raw.

</details>

---

### Q10. Your marketing team wants customer segments. What do you actually do?

<details><summary>Answer</summary>

Almost none of the work is running the algorithm.

1. **Ask what the segments are for.** Different campaigns, different pricing, different support
   tiers? The intended *use* determines which features belong in the distance — and §1.1 shows
   that the same data has several defensible clusterings, so this is not a formality.
2. **Ask how many they can act on.** If marketing can run four campaigns, $k=4$ — and §1.7 says
   your metrics were not going to identify a better $k$ anyway. This is the strongest constraint
   you will get and it is free.
3. **Engineer the features deliberately.** RFM (recency, frequency, monetary value) is the
   standard starting point. Every feature you include is a statement that it should influence
   the segmentation.
4. **Scale** (Q9). Monetary value in currency and recency in days are not comparable.
5. **Handle skew.** Spend is usually log-normal, and k-means on raw spend produces one cluster
   of whales and one of everyone else. Log-transform first.
6. **Cluster, with `n_init` at its default** (Q2). Start with k-means as a baseline.
7. **Check it is not noise** — gap statistic (§1.8) — and **check it is stable** (Q11).
8. **Profile the clusters, then name them.** Compare each cluster's feature means against the
   population. A segment you cannot describe in one sentence will not survive the meeting.
9. **Validate externally.** Do the segments differ in something you did not cluster on —
   conversion, churn, margin? That is the test that matters.

**Two failure modes to name explicitly:** k-means will not find your small high-value segment
(§1.5 — ARI 0.03 on 580/15/5), so look for it separately; and segments drift, so re-run on a
schedule and expect them to move.

</details>

---

### Q11. How do you know your clusters are real and not an artifact?

<details><summary>Answer</summary>

Four checks, in increasing order of how much they tell you.

**1. Compare against a null.** The gap statistic (§1.8) compares your inertia against uniform
data spanning the same box. On 8 independent noise samples it answered $k=1$ **100% of the
time**, where silhouette scored the same noise at 0.42. This is the check that directly asks
"is there structure at all?"

**2. Test stability under resampling.** Bootstrap the rows, re-cluster, and measure agreement
with the original partition using ARI. Real structure survives; artifacts do not. This also
catches the §1.3 problem — if different *runs* on the same data disagree, you have a local-optima
problem before you have a stability problem.

**3. Test stability under perturbation.** Add small noise, drop a feature, drop 10% of rows. A
clustering that reorganises completely when one feature is removed was that feature's story.

**4. Validate on something external.** Do the clusters differ on a variable you did *not*
cluster on? This is the strongest evidence available and it is the only one that speaks to
whether the clusters *mean* anything, as opposed to merely existing.

**What does not count as evidence:** a good silhouette score (§1.8 — noise scores 0.42), a
convincing 2-D scatter plot (a projection can manufacture apparent separation, and t-SNE
especially so — NB-09 §3.2), or clusters that "look sensible" once you have already named them.

</details>

---

### Q12. You cluster your customers, ship the segments, and three months later they make no sense. What happened?

<details><summary>Answer</summary>

Five candidates, roughly in order of likelihood.

1. **There were never any clusters.** The most common and least suspected. k-means always
   returns a partition (§1.8), so "we got five segments" is not evidence that five segments
   exist. If you did not run a null comparison at the time, you cannot rule this out now — and
   the symptom of no-structure is exactly this: results that do not reproduce.
2. **The clustering was unstable.** A different `random_state` would have given a different
   answer (§1.3 — 21 distinct solutions from 30 runs). If nobody fixed the seed *and* checked
   stability, the segments were partly an artifact of initialisation.
3. **Genuine drift.** Customers change, and clusters legitimately move. This is the benign
   explanation, and you can distinguish it from the others: re-cluster the *old* data and check
   you still get the old segments. If you do, it is drift; if you do not, it was never stable.
4. **The pipeline was not preserved.** Cluster centres, the scaler's means and standard
   deviations, and any PCA rotation are all part of the model. If the scaler was re-fitted on new
   data, the assignments are being computed in a different coordinate system.
5. **They never meant anything.** The segments were geometric groupings that nobody validated
   against an outcome (Q8, Q11). They "made sense" because a human named them, and names are easy
   to invent for arbitrary partitions.

**The monitoring that would have caught it:** track cluster *sizes* over time, the mean distance
from points to their assigned centre (rising = the model no longer fits), and the share of points
whose assignment changes on re-clustering. All three are cheap and need no labels.

</details>

---

## Coding challenges

### Challenge 1 — build the stability check from Q11

Turn "are these clusters real?" into a number.

1. Write `cluster_stability(X, k, n_boot=100)`: bootstrap-resample the rows, cluster each
   resample, and measure ARI between the resample's labels and the original clustering's labels
   **on the points that appear in both**.
2. Run it on `make_blobs` with 4 well-separated clusters, at k = 2, 3, 4, 5, 6. Stability should
   peak at the true k.
3. Now run the identical procedure on uniform noise. What does the stability curve look like
   when there is nothing to find? Does it have a peak anyway?
4. Compare your stability-based choice of k against silhouette and the gap statistic on both
   datasets. Which is most trustworthy, and at what compute cost?
5. Apply it to the digits data from Part 2. Does stability identify k=10?

---

### Challenge 2 — k-means as image compression

k-means is a vector quantiser, and this is its most tangible application.

1. Load any colour photograph. Reshape it to (n_pixels, 3) — every pixel is a point in RGB
   space.
2. Cluster the pixel colours with k = 2, 4, 8, 16, 32, 64 and replace each pixel with its cluster
   centre. Display the results.
3. Compute the compression ratio: you now store $k$ colours plus one index per pixel, instead of
   three bytes per pixel. Plot file-size-equivalent against visual quality.
4. At what $k$ does the image stop looking degraded? Compare that with the reconstruction
   question in NB-09 Challenge 1 — both are lossy compression, but they discard different things.
5. Use `MiniBatchKMeans` instead and compare quality and time. A megapixel image is a million
   points, which is where the mini-batch variant earns its place.

---

### Challenge 3 — make every method fail

The fastest way to understand four algorithms is to construct the data each one cannot handle.

1. Build four 2-D datasets: (a) two moons, (b) clusters of sizes 500/50/5, (c) three clusters of
   wildly different densities, (d) 1,000 uniform random points with no structure.
2. Run k-means, Ward, DBSCAN and GMM on each with a sensible k. Tabulate ARI (using the true
   labels for a–c; for (d) report what each method returns and how many clusters).
3. For each cell where a method failed, write one sentence explaining **which assumption broke**.
4. Now try to fix each failure without changing the method — feature engineering, a different
   distance, a transform. Which failures are fixable by preprocessing and which are fundamental?
5. Dataset (d) has no right answer. Which methods can express that, and what does each of the
   others return instead?

---
# Part 5 - Five datasets to practise on

| # | Dataset | Shape | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **Iris** | 150 × 4 | The whole workflow, small enough to check | ★☆☆☆☆ |
| 2 | **Wine** | 178 × 13 | Scaling (§1.6), and comparing methods | ★★☆☆☆ |
| 3 | **Mall customer segmentation** | 200 × 5 | The Q10 workflow end to end | ★★☆☆☆ |
| 4 | **Online Retail / RFM** | 500k rows | Real feature engineering, skew, scale | ★★★★☆ |
| 5 | **Single-cell RNA-seq** | ~3k × ~20k | Where clustering is the actual science | ★★★★★ |

In [ ]:
print("bundled with sklearn:\n")
print(f"  {'dataset':<16} {'rows':>6} {'cols':>6} {'true classes':>14}")
print("  " + "-" * 46)
for label, loader in [("iris", load_iris), ("wine", load_wine)]:
    b = loader()
    print(f"  {label:<16} {b.data.shape[0]:>6} {b.data.shape[1]:>6} "
          f"{len(np.unique(b.target)):>14}")
print(f"  {'digits':<16} {DIGITS.data.shape[0]:>6} {DIGITS.data.shape[1]:>6} "
      f"{len(np.unique(DIGITS.target)):>14}")

print("\nk-means on each, scaled, at the true k (ARI against the known classes):\n")
for label, loader in [("iris", load_iris), ("wine", load_wine)]:
    X_, y_ = loader(return_X_y=True)
    Xs_ = StandardScaler().fit_transform(X_)
    k = len(np.unique(y_))
    lab = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(Xs_)
    print(f"  {label:<16} ARI {adjusted_rand_score(y_, lab):.4f}   "
          f"silhouette {silhouette_score(Xs_, lab):.4f}")

print()
print("Note how modest those ARI values are even with the correct k and correct scaling.")
print("Recovering known classes by clustering is genuinely hard - the classes were defined")
print("by something other than geometric compactness.")
print()
print("Mall customers and Online Retail are downloads; loaders are in the briefs below.")

### 1. Iris — the whole workflow at a size you can check

```python
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
```

Three species, four measurements. Two of the species overlap, which makes it more honest than
it looks.

1. Cluster with k = 2, 3, 4 and compute silhouette for each. Silhouette will prefer **k=2**
   while the truth is 3 — explain why using §1.7's argument.
2. Compare ARI at k=3 with and without scaling. All four features are in centimetres, so this is
   Q9's "already commensurable" case. Does scaling help?
3. Run the gap statistic. Does it prefer 2 or 3?
4. Plot the clusters on the first two principal components (NB-09) and colour by both cluster
   and true species. Which species merge, and does that match the confusion you would predict
   from the pairplot?
5. Now do it with GMM. Overlapping classes are exactly the soft-assignment case (§3.3) — how many
   points get a maximum probability below 0.8, and are they the ones on the boundary?

---

### 2. Wine — scaling, and a fair method comparison

```python
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True)
```

13 chemical measurements on very different scales. §1.6 uses it for exactly that.

1. Reproduce §1.6's raw-vs-scaled result, then look at which single feature dominates the raw
   clustering. (NB-09 §1.5 found the same feature dominating PC1 — the same cause.)
2. Compare k-means, Ward, DBSCAN and GMM at k=3 on scaled data. Report ARI **and** silhouette.
   Do they rank the methods the same way?
3. Try `RobustScaler` and `MinMaxScaler`. Does the choice of scaler matter as much as the
   choice to scale?
4. Run PCA to 2 components first, then cluster. Better or worse than clustering all 13? Explain
   using NB-07 §1.5.
5. Profile the clusters: for each, which chemicals are above or below the population mean? Write
   a one-sentence description of each cluster, then check it against the true cultivar.

---

### 3. Mall customer segmentation — the Q10 workflow

```python
# widely mirrored; e.g.
# https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv
```

200 customers with age, income and a spending score. The canonical teaching dataset for
segmentation, and small enough to see everything.

1. Cluster on income and spending score only. The elbow will be unusually clear here — note that
   this is *not* typical (§1.7).
2. Now add age. Do the segments change? Which features you include **is** the modelling
   decision.
3. Profile and name each segment in one sentence (Q10 step 8).
4. Run the gap statistic and a stability check (Challenge 1). Do the segments survive?
5. This dataset is famously clean. Add realistic noise — 10% of rows with a shuffled spending
   score — and see how much survives. That gap is the difference between a tutorial and a job.

---

### 4. Online Retail — RFM segmentation on real transactions

```python
# UCI Online Retail: https://archive.ics.uci.edu/dataset/352/online+retail
# ~540k transaction rows -> aggregate to one row per customer
```

Real, messy, and the feature engineering is most of the work.

1. Aggregate transactions to per-customer **recency, frequency, monetary value**. Handle
   returns (negative quantities), missing customer ids, and cancelled orders.
2. Plot each RFM feature. Monetary value will be extremely skewed — log-transform it and explain
   why k-means demands that (Q10 step 5).
3. Scale, then cluster. Compare k-means with GMM, and check whether the small high-value segment
   survives (§1.5 predicts k-means will lose it).
4. Profile the segments and check they differ on something you did **not** cluster on — for
   instance return rate or basket size. That is the external validation of Q8.
5. Split the data by time: cluster on the first six months, assign the next six, and measure how
   much the segments drift. This is Q12, made concrete.

---

### 5. Single-cell RNA-seq — clustering as the scientific result

```python
# pip install scanpy
import scanpy as sc
adata = sc.datasets.pbmc3k()      # ~2,700 cells x ~32,000 genes
```

Here the clusters *are* the finding — they are putative cell types — so every caveat in this
notebook has real consequences.

1. Follow the standard pipeline: filter, normalise, log-transform, select highly variable genes,
   **PCA to ~50 components** (NB-09 §5), then build a neighbour graph and cluster with Leiden.
2. Note that the field does **not** use k-means. Work out why from §3.2 and §1.5 — think about
   dimensionality and about rare cell types.
3. Vary the Leiden resolution parameter. It plays the role of $k$, and the number of "cell types"
   you discover depends on it. Sit with what that means for a published result.
4. Check stability (Challenge 1) across resamples and across resolutions.
5. Identify marker genes per cluster and check them against known cell-type markers. This is
   external validation (Q8) done properly, and it is why the field trusts these clusters at all.

---
# Part 6 - Reading the literature

Clustering's literature is unusually honest about its own difficulty — the best papers here are
about why the problem is hard, not about new algorithms.

## Start here

**1. [k-means++: The Advantages of Careful Seeding](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf)** —
David Arthur & Sergei Vassilvitskii, SODA 2007. **Free.**
> **Short, elegant, and you use it every time you call `KMeans`.** A one-paragraph change to
> initialisation, with a proof that it gets you within $O(\log k)$ of optimal before Lloyd's
> algorithm even starts. §1.4 measures the effect. A good example of a small idea with a large
> practical payoff.

**2. [A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise](https://cdn.aaai.org/KDD/1996/KDD96-037.pdf)** —
Martin Ester, Hans-Peter Kriegel, Jörg Sander & Xiaowei Xu, KDD 1996. **Free.**
> **DBSCAN.** Notable for reframing what a cluster *is* — a dense region rather than a set of
> points near a centre — which is what lets it find arbitrary shapes and label noise. Won the KDD
> test-of-time award in 2014. §3.2 is this paper.

**3. [Estimating the number of clusters in a data set via the gap statistic](https://hastie.su.domains/Papers/gap.pdf)** —
Robert Tibshirani, Guenther Walther & Trevor Hastie, JRSS-B 63(2), 2001. **Free.**
> **The one that takes "how many clusters?" seriously.** Compares your clustering against what
> you would get on structureless data of the same shape — which is why it is the only method in
> §1.7 that can answer $k=1$. §1.8 verifies it correctly detects noise on 8 out of 8 samples.
> Read it if you take one methodological idea from this notebook.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.2 — the algorithm | **Lloyd**, *Least squares quantization in PCM*, Bell Labs 1957, published IEEE Trans. Inf. Theory **1982**; and **MacQueen**, *Some methods for classification and analysis of multivariate observations*, Berkeley Symposium **1967** (the name "k-means") | 🔍 |
| 1.2 — that the global optimum is NP-hard | **Aloise, Deshpande, Hansen & Popat**, *NP-hardness of Euclidean sum-of-squares clustering*, Machine Learning 75, **2009** | 🔍 |
| 1.4 — k-means++ | **Arthur & Vassilvitskii**, SODA **2007** — [pdf](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf) | ✅ |
| 1.7 — silhouette | **Rousseeuw**, *Silhouettes: a graphical aid to the interpretation and validation of cluster analysis*, J. Comp. Appl. Math. 20, **1987** | 🔍 |
| 1.7 — Calinski–Harabasz | **Caliński & Harabasz**, *A dendrite method for cluster analysis*, Communications in Statistics 3(1), **1974** | 🔍 |
| 1.7 — Davies–Bouldin | **Davies & Bouldin**, *A Cluster Separation Measure*, IEEE TPAMI 1(2), **1979** | 🔍 |
| 1.7–1.8 — **the gap statistic** | **Tibshirani, Walther & Hastie**, JRSS-B **2001** — [pdf](https://hastie.su.domains/Papers/gap.pdf) | ✅ |
| 1.8, Q8 — **why clustering cannot be solved in general** | **Kleinberg**, *An Impossibility Theorem for Clustering*, NeurIPS **2002** — [pdf](https://papers.nips.cc/paper_files/paper/2002/hash/43e4e6a6f341e00671e123714de019a8-Abstract.html) | ✅ |
| 3.1 — Ward linkage | **Ward**, *Hierarchical Grouping to Optimize an Objective Function*, JASA 58(301), **1963** | 🔍 |
| 3.2 — DBSCAN | **Ester, Kriegel, Sander & Xu**, KDD **1996** — [pdf](https://cdn.aaai.org/KDD/1996/KDD96-037.pdf) | ✅ |
| 3.2 — HDBSCAN, the modern successor | **Campello, Moulavi & Sander**, *Density-Based Clustering Based on Hierarchical Density Estimates*, PAKDD **2013** | 🔍 |
| 3.3 — EM, which fits the GMM | **Dempster, Laird & Rubin**, *Maximum Likelihood from Incomplete Data via the EM Algorithm*, JRSS-B 39(1), **1977** | 🔍 |
| 3.3 — BIC | **Schwarz**, *Estimating the Dimension of a Model*, Annals of Statistics 6(2), **1978** | 🔍 |
| Q11 — stability as validation | **Ben-Hur, Elisseeff & Guyon**, *A stability based method for discovering structure in clustered data*, PSB **2002** — [pdf](https://psb.stanford.edu/psb-online/proceedings/psb02/benhur.pdf) | ✅ |
| Q8 — comparing partitions | **Hubert & Arabie**, *Comparing partitions*, J. Classification 2, **1985** (the adjusted Rand index) | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Kleinberg (2002), *An Impossibility Theorem for Clustering*.** It proves that no clustering
function can simultaneously satisfy three properties that all seem obviously desirable — scale
invariance, richness, and consistency. You must give one up.

That is the formal version of everything this notebook found empirically: the metrics disagree
(§1.7), the algorithms encode incompatible notions of "cluster" (§3.4), and there is no
universally correct answer to recover. Reading it converts a nagging sense that clustering is
unsatisfying into a precise understanding of *why it has to be*.

Then read **Tibshirani et al. (2001)** for the most useful practical technique here, and
**Arthur & Vassilvitskii (2007)** because it is short and you run it daily.

---
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Different clusters every run | Local optima (§1.3) | Keep `n_init=10`; set `random_state`; check stability (Q11) |
| One giant cluster and several tiny ones | Unscaled features (§1.6) | `StandardScaler` — unless features are already commensurable (Q9) |
| A small, important segment is never found | WCSS is a sum, so big clusters dominate (§1.5) | GMM, DBSCAN, or anomaly detection (NB-15) |
| Elbow plot has no elbow | Normal — inertia always decreases (§1.7) | Use several metrics; use domain knowledge; gap statistic |
| Metrics disagree about k | Normal (§1.7) — they measure different things | Treat as evidence, not answer; validate externally |
| Silhouette ~0.4 and you are pleased | Pure noise scores that (§1.8) | Compare against a null — gap statistic |
| Clusters look great, mean nothing | No external validation (Q8, Q11) | Check they predict something you did not cluster on |
| Non-convex clusters split down the middle | k-means draws linear boundaries (§3.4) | DBSCAN, spectral clustering, or single-linkage (with Q's caveat) |
| DBSCAN returns 1 cluster or all noise | `eps` wrong, or too many dimensions (§3.2) | Tune `eps` via a k-NN distance plot; reduce dimensions; try HDBSCAN |
| Single linkage worked, then broke | Chaining — a thin bridge merges clusters (§3.1) | Use Ward unless you have a specific reason |
| GMM raises or gives degenerate components | Too few points per component in high dimensions | `covariance_type="diag"`, raise `reg_covar`, reduce dimensions |
| `ValueError: n_samples=4 should be >= n_clusters=5` | k exceeds the number of points | Lower k |
| Hierarchical clustering exhausts memory | $O(n^2)$ distance matrix (§3.1) | Subsample, or use k-means / MiniBatchKMeans |
| Production assignments drift | Scaler/centres not preserved, or genuine drift (Q12) | Ship the whole pipeline; monitor cluster sizes and mean distance-to-centre |

## Checklist for shipping a clustering

Everything in the Foundations checklist, plus:

- [ ] Did I state **what the clusters are meant to mean** before running anything (§1.1)?
- [ ] Are features **scaled** — or have I deliberately decided they are commensurable (Q9)?
- [ ] Is `n_init` at its default, and is `random_state` fixed (§1.3)?
- [ ] Have I checked there is **structure at all**, against a null (§1.8)?
- [ ] Is the clustering **stable** under resampling and reseeding (Q11)?
- [ ] Did I choose k from **domain knowledge or actionability**, rather than a metric alone (Q3)?
- [ ] Have I looked at **exemplars from each cluster**, not only the summary statistics?
- [ ] Can I describe each cluster in **one sentence**?
- [ ] Do the clusters differ on something I did **not** cluster on (Q8)?
- [ ] If small clusters matter, have I confirmed k-means did not swallow them (§1.5)?
- [ ] Does the shipped artifact include the **scaler, any PCA rotation, and the centres**?
- [ ] Am I monitoring cluster sizes and mean distance-to-centre over time (Q12)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| [`pca_zero_to_hero.ipynb`](pca_zero_to_hero.ipynb) | The standard preprocessing step in front of clustering, and the other unsupervised method. §2.1 used it here. |
| `anomaly_detection_zero_to_hero.ipynb` | §1.5 showed k-means unable to find a 5-point cluster. Finding the rare thing is its own problem, with its own methods. |
| [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) | The supervised twin — same Euclidean distance, same scaling requirement, same curse of dimensionality that breaks DBSCAN in §3.2. |
| `imbalanced_classification_zero_to_hero.ipynb` | The supervised version of "the interesting group is small and the objective does not care". |

See [`ZERO_TO_HERO_PLAN.md`](../ZERO_TO_HERO_PLAN.md) for the full roster and status.